## 1 · Settings


In [ ]:
from pathlib import Path

# For the first run, attach your permitted LAV-DF input and change this if auto-detection is ambiguous.
DATA_PATH = "/kaggle/input/datasets/elin75/localized-audio-visual-deepfake-dataset-lav-df/LAV-DF"
ACTION = "train"                   # train | predict  (predict needs only BUNDLE_PATH, not LAV-DF)
PRESET = "pilot"                   # quick: 64/32/32 | pilot: 320/96/160 | study: 3200/800/1200 | full
FEATURE_MODE = "light"             # light: offline CPU | transfer: frozen XLS-R + ResNet-18 on Kaggle GPU
SEEDS = [17]                       # final comparison: [17, 29, 43]; split selection always uses DATA_SEED
RUN_FINAL_TEST = False              # True only after model/config decisions have been fixed on dev
RUN_ROBUSTNESS = False             # clean, +/-240ms feature shifts, real re-encoding, and combined
ALLOW_VARIANTS = False             # enable only where terms/consent cover the derived analyses
USE_SHIFT_CONSISTENCY = False      # paired shift augmentation + consistency; requires ALLOW_VARIANTS
FACE_MODE = "centered"             # centered: single face-framed clips | track: OpenCV frontal-face tracking
XLSR_WEIGHTS = "facebook/wav2vec2-xls-r-300m"  # or approved local model directory with config and weights
RESNET_WEIGHTS = ""                # local ImageNet ResNet-18 state dict, or "" for torchvision DEFAULT
ALLOW_WEIGHT_DOWNLOAD = False      # transfer mode: enable Internet/weight downloads OR mount local weights
PREVIOUS_OUTPUT = None             # previous attached banglaav_lavdf output root; feature caches are reusable
BUNDLE_PATH = None                 # predict mode: your exported model_bundle.zip
NEW_VIDEO = None                   # optional path to a new single-speaker MP4
EXTERNAL_CSV = None                # optional reviewed external CSV; blank segment cell = unknown
EXTERNAL_ROOT = None               # default for relative paths: directory containing EXTERNAL_CSV
BA_TFD_JSON = None                 # optional authors' fusion output JSON for the SAME LAV-DF test files
BA_TFD_LABEL = "BA-TFD"            # for imported BA_TFD_JSON: BA-TFD or BA-TFD+
MEDIA_ROOT = None                  # optional extracted videos root if metadata is stored elsewhere
ALLOW_PARTIAL_INPUT = False        # explicit: use only available release clips in a partial distribution
INDEEPFAKE_VIDEO_ROOT = None       # optional inventory; creates an unlabelled CSV for review
LIP_TO_TEXT_VIDEO_ROOT = None      # point specifically to Face_Only_Videos_with_Audio
OFFICIAL_REPO = None               # optional local clone of ControlNet/LAV-DF
OFFICIAL_CKPT = None               # optional authors' BA-TFD or BA-TFD+ Lightning checkpoint
OFFICIAL_PYTHON = None             # interpreter in a compatible isolated official environment
OFFICIAL_CONFIG = "config/batfd_default.toml"
TRAIN_GENERATOR_FAMILIES = []      # enter documented names only, consistent across all manifests
HELDOUT_EXTERNAL_FAMILIES = []     # optional zero-shot audit; empty means no unseen-generator claim
SOURCE_MAP_CSV = None              # optional trusted file,speaker_id,source_id map for LAV-DF
INPUT_BASE = Path("/kaggle/input")
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT = WORK / "banglaav_lavdf"
# Used only if the old split cannot be reconstructed from the mounted release.
PRIOR_QUICK_MANIFEST = None              # original quick selected_manifest.csv, including all three splits
PRIOR_QUICK_PROTOCOL = "b0b1269dce445950"
PRIOR_QUICK_FINGERPRINT = "ac8252b658e4af662d180bf9871a36d256ad661b6b2062fe3298c46ab7706e57"
DATA_SEED = 2026
STEP = 0.08                        # 80 ms bins; predicted boundaries quantized to this analysis grid
MAX_SECONDS = 60.0                 # explicit media guard; use short single-speaker clips
MAX_FIT_WINDOWS = 18000            # per head, sampled reproducibly from train only
EPOCHS = 16
HIDDEN = 48
THREADS = 2
MAX_DISK_CACHE_GB = 12             # guard (GiB); move completed cache to private input for later sessions
EVAL_MODEL = "robust_delay_aware"         # fixed before external inference
CODE_VERSION = "nsu-lav-1.1-lwca"

print(f"START | BanglaAV | preset={PRESET} | mode={ACTION} | final_test={RUN_FINAL_TEST}", flush=True)
print("Settings loaded:", DATA_PATH, flush=True)

##  2· Environment Checkup



In [ ]:
from time import monotonic
import contextlib, csv, hashlib, io, json, math, os, re, shutil, subprocess, sys, tarfile, tempfile, zipfile
import importlib.metadata as imeta
from collections import Counter, defaultdict
from pathlib import PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score
from threadpoolctl import threadpool_limits

OUT.mkdir(parents=True, exist_ok=True)
FFMPEG, FFPROBE = shutil.which("ffmpeg"), shutil.which("ffprobe")
if not FFMPEG or not FFPROBE:
    raise RuntimeError("A Python environment with ffmpeg and ffprobe is required.")
if ACTION not in {"train", "predict"} or FEATURE_MODE not in {"light", "transfer"}:
    raise ValueError("Check ACTION and FEATURE_MODE in Settings.")
if PRESET not in {"quick","pilot","study","full"} or not SEEDS or len(set(SEEDS)) != len(SEEDS):
    raise ValueError("Use a listed preset and distinct training seeds.")
if (RUN_ROBUSTNESS or USE_SHIFT_CONSISTENCY) and not ALLOW_VARIANTS:
    raise ValueError("Confirm permission in Settings before running shift or compression experiments.")

if BA_TFD_LABEL not in ("BA-TFD","BA-TFD+"):
    raise ValueError("BA_TFD_LABEL must name the authors’ BA-TFD or BA-TFD+ model.")
if ACTION=="train" and EXTERNAL_CSV and not RUN_FINAL_TEST:
    raise ValueError("Keep external data sealed during development. Set RUN_FINAL_TEST=True only after locking the model.")
if STEP!=.08 or MAX_SECONDS<=0 or EPOCHS<1 or HIDDEN<1 or MAX_FIT_WINDOWS<4 or THREADS<1:
    raise ValueError("This protocol uses an 80ms grid; provide positive training/runtime settings.")
if not all(type(seed) is int and seed>=0 for seed in SEEDS):
    raise ValueError("Seeds must be nonnegative integers.")

def command(argv, timeout=180):
    proc = subprocess.run([str(x) for x in argv], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
    if proc.returncode:
        raise RuntimeError(proc.stderr.decode("utf-8", "replace")[-2200:])
    return proc.stdout

def uid(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, default=str).encode()).hexdigest()

def json_ready(x):
    if isinstance(x, dict): return {str(k):json_ready(v) for k,v in x.items()}
    if isinstance(x, (tuple,list,np.ndarray)): return [json_ready(v) for v in x]
    if isinstance(x, (float,np.floating)): return float(x) if np.isfinite(x) else None
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, Path): return str(x)
    return x

@contextlib.contextmanager
def atomic_path(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name+".",suffix=".part",dir=path.parent)
    os.close(fd)
    try:
        yield Path(tmp)
        os.replace(tmp,path)
    finally:
        Path(tmp).unlink(missing_ok=True)

def write_json(path, obj):
    with atomic_path(path) as p:
        p.write_text(json.dumps(json_ready(obj), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def write_csv(path, rows):
    with atomic_path(path) as p:
        pd.DataFrame(rows).to_csv(p, index=False, encoding="utf-8-sig")

def tokensafe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+","_",str(name))[:75] or "group"

def safe_key(path):
    raw = str(path).replace("\\","/")
    pp = PurePosixPath(raw)
    if pp.is_absolute() or ".." in pp.parts or re.match(r"^[A-Za-z]:",raw):
        raise ValueError(f"Expected a safe relative media path, got {path!r}.")
    return pp.as_posix()

def flag(x):
    if type(x) is bool: return x
    if isinstance(x,(int,str)) and str(x).lower() in {"0","1","true","false"}:
        return str(x).lower() in {"1","true"}
    raise ValueError(f"Invalid modality flag: {x!r}")

def merge_spans(spans):
    result = []
    for s,e in sorted(spans):
        if result and s <= result[-1][1] + 1e-7:
            result[-1][1] = max(result[-1][1],e)
        else:
            result.append([float(s),float(e)])
    return result

def checked_spans(obj,duration):
    if not isinstance(obj,list): raise ValueError("Intervals must be a list of [start,end] seconds.")
    valid=[]
    for pair in obj:
        if not isinstance(pair,(tuple,list)) or len(pair)!=2:
            raise ValueError(f"Invalid interval {pair!r}")
        if not all(isinstance(q,(float,int)) and not isinstance(q,bool) for q in pair):
            raise ValueError(f"Numeric seconds required: {pair!r}")
        s,e = map(float,pair)
        if not np.isfinite([s,e]).all() or s < -1e-5 or not s<e or e > duration+0.081:
            raise ValueError(f"Interval {pair} outside duration {duration}. Check units.")
        s,e=max(0.,s),min(e,duration)
        if e>s: valid.append([s,e])
    return merge_spans(valid)

def shown(frame):
    try: display(frame)
    except NameError: print(frame.to_string(index=False))

def version(pkg):
    try: return imeta.version(pkg)
    except imeta.PackageNotFoundError: return None

FFVERSION=command([FFMPEG,"-version"]).decode().splitlines()[0]
RUNTIME={k:version(k) for k in ("numpy","pandas","scikit-learn","torch","torchvision","transformers","opencv-python","pillow")}
RUNTIME.update(python=sys.version.split()[0],ffmpeg=FFVERSION)
print("Runtime:",RUNTIME["python"],"|",FFVERSION,"| output:",OUT)

## 3 · Metadata And Different test


In [ ]:
# Streams the records of either metadata file without retaining transcripts and word timestamps.
def json_records(binary):
    from json import JSONDecoder
    decoder=JSONDecoder()
    stream=io.TextIOWrapper(binary,encoding="utf-8-sig")
    buf=""; pos=0; end=False
    def more():
        nonlocal buf,pos,end
        fresh=stream.read(256*1024)
        buf=buf[pos:]+fresh; pos=0; end=not fresh
    def peek():
        nonlocal pos
        while True:
            while pos<len(buf) and buf[pos].isspace(): pos+=1
            if pos<len(buf): return buf[pos]
            if end: return ""
            more()
    try:
        if peek()!="[": raise ValueError("Expected the official top-level JSON array.")
        pos+=1
        if peek()=="]":
            pos+=1
            if peek(): raise ValueError("Unexpected data after closing JSON array.")
            return
        while True:
            while True:
                try:
                    entry,following=decoder.raw_decode(buf,pos)
                    pos=following
                    break
                except json.JSONDecodeError:
                    if end: raise
                    more()
            if not isinstance(entry,dict): raise ValueError("Metadata record must be an object.")
            yield entry
            token=peek()
            if token=="]":
                pos+=1
                if peek(): raise ValueError("Unexpected data after closing JSON array.")
                return
            if token!=",": raise ValueError("Missing comma between metadata records.")
            pos+=1
            if not peek(): raise ValueError("Truncated metadata file.")
    finally:
        stream.detach()

def choose_metadata(names):
    names=list(names)
    for filename in ("metadata.min.json","metadata.json"):
        options=[n for n in names if PurePosixPath(n).name.lower()==filename]
        if len(options)==1: return options[0]
        if len(options)>1: raise ValueError(f"Multiple {filename} files. Point DATA_PATH to one extracted metadata file.")
    raise FileNotFoundError("metadata.min.json or metadata.json not found.")

def auto_dataset():
    if not INPUT_BASE.is_dir(): raise FileNotFoundError("Set DATA_PATH to your LAV-DF folder or archive.")
    folders=set(); archives=[]
    for parent,dirs,files in os.walk(INPUT_BASE):
        if any(n.lower() in {"metadata.min.json","metadata.json"} for n in files):
            folders.add(Path(parent)); dirs[:]=[]; continue
        dirs[:]=[d for d in dirs if d.lower() not in {"train","training","val","dev","test","validation"}]
        for name in files:
            if re.search(r"lav[-_. ]?df",name,re.I) and name.lower().endswith((".tar",".zip",".tar.gz",".tgz")):
                archives.append(Path(parent)/name)
    options=sorted(folders) or sorted(archives)
    if len(options)!=1:
        raise ValueError(f"Set DATA_PATH explicitly. Candidates: {[str(p) for p in options[:12]]}")
    return options[0]

class LavSource:
    """A folder or one official archive; only requested video bytes are materialized."""
    def __init__(self,path):
        p=Path(path).expanduser().resolve()
        if not p.exists(): raise FileNotFoundError(p)
        self.path=p
        self.kind="folder"; self.entries={}; self.members={}; self.z=None
        self.staged={}; self.temp=contextlib.ExitStack()
        if p.is_dir() or p.suffix.lower()==".json":
            if p.is_file(): meta=p
            else:
                direct=[v.name for v in p.iterdir() if v.is_file()]
                if any(v.lower() in {"metadata.json","metadata.min.json"} for v in direct):
                    meta=p/choose_metadata(direct)
                else:
                    matches=[str(v.relative_to(p)) for v in p.rglob("metadata*.json")
                             if v.name.lower() in {"metadata.json","metadata.min.json"}]
                    meta=p/choose_metadata(matches)
            self.base=Path(MEDIA_ROOT).resolve() if MEDIA_ROOT else meta.parent; self.meta=meta
            return
        self.signature=(p.stat().st_size,p.stat().st_mtime_ns)
        if zipfile.is_zipfile(p):
            self.kind="zip"; self.z=zipfile.ZipFile(p)
            for m in self.z.infolist():
                if not m.is_dir(): self._add(m.filename,m)
        elif tarfile.is_tarfile(p):
            with p.open("rb") as b: magic=b.read(6)
            self.kind="ctar" if magic.startswith((b"\x1f\x8b",b"BZh",b"\xfd7zXZ")) else "tar"
            with tarfile.open(p,"r:*") as tar:
                for m in tar:
                    if m.isfile(): self._add(m.name,(m.offset_data,m.size))
        else: raise ValueError("Use extracted data, .tar, .tar.gz or .zip.")
        self.meta=choose_metadata(self.entries)
        self.base=PurePosixPath(self.meta).parent

    def _add(self,name,info):
        if PurePosixPath(name).suffix.lower() not in {".mp4",".mkv",".avi",".mov",".json"}: return
        normalized=safe_key(name)
        if normalized in self.entries: raise ValueError(f"Archive has duplicate path: {normalized}")
        self.entries[normalized]=info; self.members[normalized]=name

    @contextlib.contextmanager
    def metadata(self):
        if self.kind=="folder":
            with self.meta.open("rb") as f: yield f
        elif self.kind=="zip":
            with self.z.open(self.entries[self.meta]) as f: yield f
        elif self.kind=="tar":
            # Do not let the metadata reader see the next TAR member.
            with tarfile.open(self.path,"r:") as tar:
                with tar.extractfile(self.members[self.meta]) as f: yield f
        else:
            with tarfile.open(self.path,"r:*") as tar:
                with tar.extractfile(self.members[self.meta]) as f: yield f

    def locate(self,key):
        k=safe_key(key)
        if self.kind=="folder":
            p=(self.base/k).resolve()
            if not p.is_relative_to(self.base) or not p.is_file(): raise FileNotFoundError(p)
            return p
        k=safe_key(str(self.base/k))
        if k not in self.entries: raise FileNotFoundError(k)
        return k

    def ident(self,loc):
        if self.kind=="folder":
            st=loc.stat(); return [str(loc),st.st_size,st.st_mtime_ns]
        m=self.entries[loc]
        info=[m.CRC,m.file_size] if self.kind=="zip" else list(m)
        return [self.path.name,self.signature,loc,info]

    def stage(self,locs):
        # Compressed TAR has no random access. Extract just selected members in one pass.
        if self.kind!="ctar": return
        need=set(locs)-self.staged.keys()
        if not need: return
        dest=Path(self.temp.enter_context(tempfile.TemporaryDirectory(prefix="lav-sel-")))
        size=sum(self.entries[k][1] for k in need)
        if size>shutil.disk_usage(dest).free*.85: raise OSError("Selected members exceed scratch space. Lower PRESET or mount an extracted folder.")
        with tarfile.open(self.path,"r|*") as tar:
            for m in tar:
                k=safe_key(m.name)
                if m.isfile() and k in need:
                    target=dest/(uid(k)+PurePosixPath(k).suffix)
                    with tar.extractfile(m) as src, target.open("wb") as out: shutil.copyfileobj(src,out)
                    self.staged[k]=target; need.remove(k)
                    if not need: break
        if need: raise FileNotFoundError(f"Missing selected archive members: {len(need)}")

    @contextlib.contextmanager
    def media(self,loc):
        if self.kind=="folder": yield loc; return
        if self.kind=="ctar": yield self.staged[loc]; return
        with tempfile.TemporaryDirectory(prefix="lav-clip-") as t:
            path=Path(t)/("clip"+PurePosixPath(loc).suffix)
            with path.open("wb") as out:
                if self.kind=="zip":
                    with self.z.open(self.entries[loc]) as f: shutil.copyfileobj(f,out)
                else:
                    off,size=self.entries[loc]
                    with self.path.open("rb") as f:
                        f.seek(off); remaining=size
                        while remaining:
                            chunk=f.read(min(1024*1024,remaining))
                            if not chunk: raise EOFError(loc)
                            out.write(chunk); remaining-=len(chunk)
            yield path

    def close(self):
        if self.z: self.z.close()
        self.temp.close()

def lav_rows(source):
    result = []; seen = set(); rejected = []
    extra = {}
    if SOURCE_MAP_CSV:
        for row in read_rows(SOURCE_MAP_CSV):
            key = safe_key(row["file"])
            if key in extra: raise ValueError("Duplicate file in SOURCE_MAP_CSV: " + key)
            extra[key] = row
    split_alias = {"train": "train", "dev": "dev", "val": "dev", "validation": "dev", "test": "test"}
    with source.metadata() as f:
        for i, r in enumerate(json_records(f)):
            if i and i % 25000 == 0:
                print(f"  Metadata: {i:,} records read", flush=True)
            try:
                key = safe_key(r["file"]); split = split_alias[str(r["split"]).lower()]
                dur = float(r["duration"])
                if not np.isfinite(dur) or dur <= 0: raise ValueError("Invalid duration")
                A = flag(r["modify_audio"]); V = flag(r["modify_video"])
                periods = checked_spans(r["fake_periods"], dur)
                if bool(periods) != (A or V): raise ValueError("Modality flags disagree with fake_periods.")
                if key in seen: raise ValueError("Duplicate file")
                seen.add(key); aux = extra.get(key, {})
                orig = r.get("original")
                group = str(aux.get("source_id") or (orig if isinstance(orig, str) and orig.strip() else key)).strip()
                speaker = str(aux.get("speaker_id") or r.get("speaker_id") or "").strip()
                if not speaker:
                    match = re.search(r"(?:^|[/\\])((?:id)\d{4,})(?:[/\\]|$)", group)
                    speaker = match.group(1) if match else ""
                kind = ("real", "audio", "visual", "both")[int(A) + 2 * int(V)]
                result.append(dict(file=key, split=split, duration=dur, a=periods if A else [], v=periods if V else [],
                                   label=int(A or V), kind=kind, group=group, speaker=speaker,
                                   group_verified=bool(aux.get("source_id") or (isinstance(orig, str) and orig.strip()) or not (A or V)),
                                   self_check=(r.get("fixture_designation") == "generated_self_check"), original=orig, video_frames=r.get("video_frames"), audio_frames=r.get("audio_frames"),
                                   audio_channels=r.get("audio_channels"), n_fakes=r.get("n_fakes", len(periods))))
            except (KeyError, TypeError, ValueError) as err:
                rejected.append(dict(metadata_row=i+1, file=str(r.get("file", "")), split=str(r.get("split", "")),
                         reason=f"{type(err).__name__}: {err}", duration=r.get("duration"), n_fakes=r.get("n_fakes"),
                         modify_audio=r.get("modify_audio"), modify_video=r.get("modify_video"),
                         fake_periods=json.dumps(r.get("fake_periods"))))
    if rejected:
        print("Metadata rows quarantined:", len(rejected), "(see metadata_exclusions.csv).")
    needed = {"train", "dev", "test"} if RUN_FINAL_TEST else {"train", "dev"}
    if not needed.issubset({r["split"] for r in result}):
        raise ValueError(f"Missing official splits {sorted(needed)}.")
    # Apply the submitted run’s split-conflict policy and preserve its row-level audit.
    for attr in ("group", "speaker"):
        found = {}; conflict = set()
        for r in result:
            val = r[attr]
            if not val: continue
            if val in found and found[val] != r["split"]: conflict.add(val)
            else: found[val] = r["split"]
        if conflict:
            for r in result:
                if r[attr] in conflict:
                    rejected.append(dict(file=r["file"], split=r["split"], reason="cross_split_"+attr, identity=r[attr]))
            before = len(result)
            result = [r for r in result if not r[attr] or r[attr] not in conflict]
            print(f"Filtered {before - len(result)} records with cross-split {attr}.")
    return result, rejected

def read_rows(path):
    with Path(path).open(encoding="utf-8-sig",newline="") as f: return list(csv.DictReader(f))

def picked(rows,split,limit):
    group=[r for r in rows if r["split"]==split]
    if limit is None or len(group)<=limit: return sorted(group,key=lambda r:r["file"])
    # Fixed proportional stratification, at least one from every available condition.
    buckets=defaultdict(list)
    for r in group: buckets[r["kind"]].append(r)
    if limit<len(buckets): raise ValueError("Split sample cap is too small.")
    ideal={k:limit*len(v)/len(group) for k,v in buckets.items()}
    n={k:min(len(v),max(1,int(ideal[k]))) for k,v in buckets.items()}
    while sum(n.values())<limit:
        k=max((k for k in n if n[k]<len(buckets[k])),key=lambda k:(ideal[k]-n[k],k)); n[k]+=1
    while sum(n.values())>limit:
        k=max((k for k in n if n[k]>1),key=lambda k:(n[k]-ideal[k],k)); n[k]-=1
    return sorted([r for k,b in buckets.items() for r in sorted(b,key=lambda x:uid([DATA_SEED,x["file"]]))[:n[k]]],key=lambda r:r["file"])

def prior_manifest_digest(rows):
    parts = [(r["file"], r["split"], r["a"], r["v"], r["duration"], r["group"], r["speaker"]) for r in rows]
    return uid(parts)


def reserve_previous_test(rows):
    # Verify the identity of the original 64/32/32 run before treating any test pool as fresh.
    splits = ("train", "dev", "test")
    if PRIOR_QUICK_MANIFEST:
        records = read_rows(PRIOR_QUICK_MANIFEST)
        grouped = {s: [] for s in splits}
        for line in records:
            grouped[line["split"]].append(dict(file=safe_key(line["file"]), split=line["split"],
                    a=json.loads(line["audio_segments"]), v=json.loads(line["visual_segments"]),
                    duration=float(line["duration_s"]), group=line["source_id"], speaker=line["speaker_id"]))
        for subset in grouped.values():
            subset.sort(key=lambda r: r["file"])
        source = "attached_prior_manifest"
    else:
        grouped = {s: picked(rows, s, n) for s, n in zip(splits, (64, 32, 32))}
        source = "reconstructed_from_release"
    combined = [r for s in splits for r in grouped[s]]
    verified = (prior_manifest_digest(combined) == PRIOR_QUICK_FINGERPRINT and
                [len(grouped[s]) for s in splits] == [64, 32, 32])
    audit = dict(quick_protocol=PRIOR_QUICK_PROTOCOL, original_manifest_sha256=PRIOR_QUICK_FINGERPRINT,
                 verified=verified, identification=source)
    if not verified:
        msg = ("Prior quick manifest did not match the submitted checkpoint. For a final test, attach the original "
               "runs/" + PRIOR_QUICK_PROTOCOL + "_a7923c02/selected_manifest.csv and set PRIOR_QUICK_MANIFEST. "
               "Check the input version and keep the original DATA_SEED=2026 for automatic identification.")
        if RUN_FINAL_TEST:
            raise ValueError(msg)
        print("Development only. " + msg)
        audit.update(test_pool_available=False, reason="original_manifest_not_verified")
        return [r for r in rows if r["split"] != "test"], [], audit

    exposed = grouped["test"]
    old_ids = {r["file"] for r in exposed}
    # A later, more complete source map may resolve additional relations for these same clips.
    linked = exposed + [r for r in rows if r["file"] in old_ids]
    groups = {r["group"] for r in linked if r["group"]}
    speakers = {r["speaker"] for r in linked if r["speaker"]}
    kept = []
    excluded = []
    for r in rows:
        causes = []
        if r["file"] in old_ids: causes.append("previous_test_clip")
        if r["group"] and r["group"] in groups: causes.append("shared_source_id")
        if r["speaker"] and r["speaker"] in speakers: causes.append("shared_speaker_id")
        if causes:
            excluded.append(dict(file=r["file"], split=r["split"], source_id=r["group"],
                                 speaker_id=r["speaker"], reason="|".join(causes)))
        else:
            kept.append(r)
    missing = sorted(old_ids - {r["file"] for r in rows})
    audit.update(test_pool_available=True, previously_seen_test_ids=len(old_ids),
                 prior_ids_now_absent=missing, excluded_by_split=dict(Counter(r["split"] for r in excluded)),
                 available_speaker_ids=len(speakers), remaining_test_pool=sum(r["split"]=="test" for r in kept))
    print("Verified previous quick test:", len(old_ids), "IDs. Reserved rows including mapped relatives:", len(excluded))
    return kept, excluded, audit

def check_input_path(configured):
    if configured and Path(configured).exists():
        return str(Path(configured))
    old_default = "/kaggle/input/datasets/elin75/localized-audio-visual-deepfake-dataset-lav-df/LAV-DF"
    if configured and str(configured) != old_default:
        raise FileNotFoundError(f"DATA_PATH does not exist: {configured}. Copy the path shown in Kaggle Input.")
    try:
        return str(auto_dataset())
    except (FileNotFoundError, ValueError) as err:
        raise FileNotFoundError(
            "LAV-DF input not found or ambiguous. In Kaggle choose Add Input > Datasets > "
            "Localized Audio Visual DeepFake Dataset (elin75). Copy the LAV-DF folder or metadata.min.json path "
            f"to DATA_PATH and run again. Details: {err}"
        ) from err

if ACTION == "train":
    print("[3/11] Checking the Kaggle input path ...", flush=True)
    DATA_PATH = check_input_path(DATA_PATH)
    print("[3/11] READY | LAV-DF input:", DATA_PATH, flush=True)
else:
    print("[3/11] READY | Prediction uses the attached model bundle.", flush=True)

## 5 · Audio ANd Face feature



In [ ]:
def digest_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1048576),b""): h.update(block)
    return h.hexdigest()

def aggbin(x,stamp,edges):
    # Aggregate samples whose timestamp lies in a bin. When empty, use nearest available sample.
    x=np.asarray(x,np.float32); stamp=np.asarray(stamp)
    if x.ndim==1: x=x[:,None]
    left=np.searchsorted(stamp,edges[:-1]); right=np.searchsorted(stamp,edges[1:])
    cs=np.concatenate([np.zeros((1,x.shape[1]),np.float64),np.cumsum(x,axis=0,dtype=np.float64)])
    out=(cs[right]-cs[left])/np.maximum(right-left,1)[:,None]
    empty=left==right
    if empty.any():
        mid=(edges[:-1]+edges[1:])[empty]/2
        ids=np.searchsorted(stamp,mid).clip(0,len(stamp)-1)
        prev=(ids-1).clip(0)
        ids=np.where(abs(stamp[prev]-mid)<abs(stamp[ids]-mid),prev,ids)
        out[empty]=x[ids]
    return out.astype(np.float32)

def probe(path):
    raw=json.loads(command([FFPROBE,"-v","error","-show_entries","format=duration:stream=codec_type,start_time,duration","-of","json",path]))
    streams=raw.get("streams",[])
    a=next((x for x in streams if x.get("codec_type")=="audio"),None)
    v=next((x for x in streams if x.get("codec_type")=="video"),None)
    if not a or not v: raise ValueError("An audio track and a visual track are both required.")
    def number(v,default):
        try: x=float(v); return x if np.isfinite(x) else default
        except (ValueError,TypeError): return default
    av,vv=number(a.get("start_time"),0.),number(v.get("start_time"),0.)
    base=min(av,vv)
    dur=number(raw.get("format",{}).get("duration"),0.)
    # Media buffers stay bounded even if optional duration metadata is missing.
    if dur>MAX_SECONDS+.25: raise ValueError(f"{dur:.2f}s > MAX_SECONDS; use a shorter approved single-speaker clip.")
    return av-base,vv-base

def decoded(path):
    sa,sv=probe(path)
    prefix=[FFMPEG,"-nostdin","-v","error","-threads","1","-i",path,"-t",str(MAX_SECONDS+.32)]
    ab=command(prefix+["-map","0:a:0","-af","asetpts=PTS-STARTPTS,aresample=16000:async=1:first_pts=0",
                       "-ac","1","-f","f32le","pipe:1"])
    vb=command(prefix+["-map","0:v:0","-vf","setpts=PTS-STARTPTS,fps=25:start_time=0,scale=144:144",
                       "-threads","1","-pix_fmt","rgb24","-f","rawvideo","pipe:1"])
    audio=np.frombuffer(ab,"<f4"); video=np.frombuffer(vb,np.uint8).reshape(-1,144,144,3)
    if len(audio)<400 or len(video)<2: raise ValueError("Clip too short or stream failed to decode.")
    if len(audio)/16000>MAX_SECONDS+.25 or len(video)/25>MAX_SECONDS+.25:
        raise ValueError("Actual media exceeds configured duration guard.")
    return audio,video,sa,sv

def mouth_sequence(frames,policy):
    if policy not in {"centered","track"}: raise ValueError("FACE_MODE must be centered or track.")
    cascade=None
    if policy=="track":
        import cv2
        cascade=cv2.CascadeClassifier(str(Path(cv2.data.haarcascades)/"haarcascade_frontalface_default.xml"))
        if cascade.empty(): raise RuntimeError("OpenCV face cascade unavailable.")
    # centered expects the displayed video to contain one large, central, already framed face.
    boxes=[]; hits=0; checks=0; last=None
    if cascade is not None:
        for t in range(0,len(frames),5):
            gray=cv2.cvtColor(frames[t],cv2.COLOR_RGB2GRAY)
            bb=cascade.detectMultiScale(gray,scaleFactor=1.1,minNeighbors=4,minSize=(24,24))
            checks+=1
            if len(bb):
                if last is None: best=max(bb,key=lambda q:int(q[2])*int(q[3]))
                else:
                    cx,cy=last[:2]+last[2:]/2
                    best=min(bb,key=lambda q:(q[0]+q[2]/2-cx)**2+(q[1]+q[3]/2-cy)**2)
                last=np.asarray(best,float); hits+=1; boxes.append((t,last.copy()))
        if not boxes or hits/checks < .25:
            raise ValueError("Face tracking coverage <25%. Inspect the face framing; report exclusions.")
    patches=[]; xp=np.array([t for t,_ in boxes]) if boxes else None
    for i,frame in enumerate(frames):
        if boxes:
            # Linear interpolation between detected faces; carry nearest box at the ends.
            coords=[np.interp(i,xp,[b[j] for _,b in boxes]) for j in range(4)]
            x,y,w,h=coords
            x0,x1=int(x+.15*w),int(x+.85*w); y0,y1=int(y+.52*h),int(y+.94*h)
        else:
            x0,x1,y0,y1=25,119,72,132
        region=frame[max(0,y0):min(144,y1),max(0,x0):min(144,x1)]
        if not region.size: raise ValueError("Empty lower-face region.")
        patches.append(np.asarray(Image.fromarray(region).resize((48,48),Image.Resampling.BILINEAR)))
    return np.stack(patches), (hits/checks if checks else -1.)

def light_audio(samples,offset,edges):
    win,hop,nfft=400,320,512
    audio=np.pad(samples,(win//2,win//2))
    blocks=np.lib.stride_tricks.sliding_window_view(audio,win)[::hop]
    energy=np.sqrt((blocks**2).mean(1)+1e-10)
    spec=(np.abs(np.fft.rfft(blocks*np.hanning(win),n=nfft))**2).astype("f")
    hz=np.fft.rfftfreq(nfft,1/16000)
    mel=lambda f:2595*np.log10(1+f/700)
    knots=700*(10**(np.linspace(mel(40),mel(7900),26)/2595)-1)
    bank=np.maximum(0,np.minimum((hz[None,:]-knots[:-2,None])/(knots[1:-1,None]-knots[:-2,None]),
                                 (knots[2:,None]-hz[None,:])/(knots[2:,None]-knots[1:-1,None])))
    lm=np.log(np.maximum(spec@bank.astype("f").T,1e-7))
    zcr=(np.diff(np.signbit(blocks),axis=1)!=0).mean(1)
    flat=np.exp(np.log(spec+1e-8).mean(1))/(spec.mean(1)+1e-8)
    stamp=offset+np.arange(len(blocks))*hop/16000
    energy_bin=aggbin(np.log(energy+1e-5),stamp,edges)
    feature=aggbin(np.column_stack([lm,zcr,flat,np.log(energy+1e-5)]),stamp,edges)
    return np.column_stack([feature,np.diff(feature,axis=0,prepend=feature[:1])]).astype("f"),energy_bin[:,0]

def light_visual(patches,offset,edges):
    gray=(patches.astype("f")@np.array([.299,.587,.114],dtype="f"))/255
    motion=np.abs(np.diff(gray,axis=0,prepend=gray[:1]))
    def pool(x): return x.reshape(len(x),4,12,4,12).mean((2,4)).reshape(len(x),16)
    gx=np.abs(np.diff(gray,axis=2)).mean((1,2)); gy=np.abs(np.diff(gray,axis=1)).mean((1,2))
    vf=np.column_stack([pool(gray),pool(motion),gx,gy,gray.std((1,2))])
    stamp=offset+(np.arange(len(gray))+.5)/25
    return aggbin(vf,stamp,edges),aggbin(motion.mean((1,2)),stamp,edges)[:,0]

class Encoder:
    def __init__(self,mode,face,weights_path=None,resnet_path=None):
        self.mode=mode; self.face=face
        self.spec=dict(code=CODE_VERSION,mode=mode,face=face,dt=STEP,ffmpeg=FFVERSION)
        if face=="track":
            import cv2
            self.spec["cv2"]=cv2.__version__
        if mode=="light": return
        if mode!="transfer": raise ValueError(mode)
        import torch
        from transformers import AutoFeatureExtractor, Wav2Vec2Model
        from torchvision.models import resnet18, ResNet18_Weights
        self.torch=torch; self.device="cuda" if torch.cuda.is_available() else "cpu"
        model_id=str(weights_path or XLSR_WEIGHTS)
        opts=dict(local_files_only=not ALLOW_WEIGHT_DOWNLOAD)
        self.front=AutoFeatureExtractor.from_pretrained(model_id,**opts)
        self.speech=Wav2Vec2Model.from_pretrained(model_id,**opts).to(self.device).eval()
        if resnet_path:
            vision=resnet18(weights=None)
            vision.load_state_dict(torch.load(str(resnet_path),map_location="cpu",weights_only=True))
            vision_id=digest_file(resnet_path)
        else:
            w=ResNet18_Weights.IMAGENET1K_V1
            weightfile=Path(torch.hub.get_dir())/"checkpoints"/Path(w.url).name
            if not ALLOW_WEIGHT_DOWNLOAD and not weightfile.is_file():
                raise RuntimeError("Mount the public ResNet-18 ImageNet state dict or explicitly enable its download.")
            vision=resnet18(weights=w); vision_id=Path(w.url).name
        vision.fc=torch.nn.Identity(); self.vision=vision.to(self.device).eval()
        cfg=self.speech.config
        stride=1; receptive=1
        for k,s in zip(cfg.conv_kernel,cfg.conv_stride):
            receptive+=(k-1)*stride; stride*=s
        self.stride,self.receptive=stride,receptive
        if Path(model_id).is_dir():
            files=sorted(p for p in Path(model_id).iterdir() if p.suffix in {".safetensors",".bin"} and p.is_file())
            if not files: raise ValueError("No local speech model weights found.")
            speech_id=uid([(p.name,digest_file(p)) for p in files]+[("config",digest_file(Path(model_id)/"config.json"))])
        else: speech_id=getattr(cfg,"_commit_hash",None) or model_id
        self.spec.update(speech=speech_id,vision=vision_id,torch=str(torch.__version__),
                         transformers=version("transformers"),torchvision=version("torchvision"))
        rng=np.random.default_rng(826)
        self.aproj=(rng.standard_normal((cfg.hidden_size,64))/8).astype("f")
        self.vproj=(rng.standard_normal((512,48))/np.sqrt(48)).astype("f")
        print("Frozen pretrained encoders:",self.device)
    def add_transfer(self,samples,patches,sa,sv,edges,A,V):
        torch=self.torch
        x=self.front(samples,sampling_rate=16000,return_tensors="pt").input_values.to(self.device)
        if x.shape[1]<self.receptive: x=torch.nn.functional.pad(x,(0,self.receptive-x.shape[1]))
        with torch.inference_mode():
            h=self.speech(x).last_hidden_state[0].float().cpu().numpy()@self.aproj
            at=sa+(np.arange(len(h))*self.stride+(self.receptive-1)/2)/16000
            ix=np.rint(((edges[:-1]+edges[1:])/2-sv)*25-.5).astype(int).clip(0,len(patches)-1)
            vec=[]
            for j in range(0,len(ix),32):
                v=torch.from_numpy(patches[ix[j:j+32]].copy()).permute(0,3,1,2).float().to(self.device)/255
                v=torch.nn.functional.interpolate(v,size=(224,224),mode="bilinear",align_corners=False)
                mean=v.new_tensor([.485,.456,.406])[None,:,None,None]; std=v.new_tensor([.229,.224,.225])[None,:,None,None]
                vec.append(self.vision((v-mean)/std).cpu().numpy()@self.vproj)
        return np.column_stack([A,aggbin(h,at,edges)]).astype("f"),np.column_stack([V,np.vstack(vec)]).astype("f")
    def extract(self,path,expected=None):
        wave,frames,sa,sv=decoded(path)
        start=max(sa,sv)
        end=min(sa+len(wave)/16000,sv+len(frames)/25)
        if expected is not None:
            if abs(end-expected)>max(.32,.03*expected) or start>.32:
                raise ValueError(f"Metadata/media timeline disagreement: expected {expected:.3f}; both streams {start:.3f}..{end:.3f}.")
            end=min(end,expected)
        if end-start<.16: raise ValueError("Too little overlapping audio and video.")
        edges=np.arange(start,end-1e-7,STEP); edges=np.r_[edges,end]
        patches,face_fraction=mouth_sequence(frames,self.face)
        A,energy=light_audio(wave,sa,edges); V,motion=light_visual(patches,sv,edges)
        if self.mode=="transfer": A,V=self.add_transfer(wave,patches,sa,sv,edges,A,V)
        return dict(A=A,V=V,ap=energy.astype("f"),vp=motion.astype("f"),edges=edges,face=np.asarray(face_fraction))

@contextlib.contextmanager
def compressed(path):
    with tempfile.TemporaryDirectory(prefix="lav-reencode-") as tmp:
        result=Path(tmp)/"compressed.mp4"
        command([FFMPEG,"-nostdin","-v","error","-y","-copyts","-start_at_zero","-i",path,
                 "-map","0:v:0","-map","0:a:0","-c:v","libx264","-crf","35","-preset","veryfast","-threads","1",
                 "-c:a","aac","-b:a","48k",result])
        yield result

class FeatureCache:
    def __init__(self,encoder):
        self.encoder=encoder
        self.folder=OUT/"features"/uid(encoder.spec)[:20]; self.folder.mkdir(parents=True,exist_ok=True)
        self.previous=(Path(PREVIOUS_OUTPUT)/"features"/self.folder.name) if PREVIOUS_OUTPUT else None
        self.size=sum(p.stat().st_size for p in self.folder.glob("*.npz"))
    def get(self,row,source=None,codec=False):
        if source is not None:
            loc=source.locate(row["file"]); ident=source.ident(loc); op=lambda:source.media(loc)
        else:
            loc=Path(row["path"]); st=loc.stat(); ident=[str(loc.resolve()),st.st_size,st.st_mtime_ns]
            op=lambda:contextlib.nullcontext(loc)
        name=uid([ident,row.get("duration"),bool(codec)])+".npz"
        target=self.folder/name
        for p in (target, self.previous/name if self.previous else None):
            if p and p.is_file():
                try:
                    with np.load(p,allow_pickle=False) as archive:
                        f={k:archive[k] for k in ("A","V","ap","vp","edges","face")}
                    n=len(f["edges"])-1
                    if n<1 or any(len(f[k])!=n for k in ("A","V","ap","vp")) or not np.all(np.diff(f["edges"])>0):
                        raise ValueError("Stale or invalid cache shape")
                    return f
                except (OSError, ValueError, EOFError, KeyError, zipfile.BadZipFile):
                    if p==target:
                        self.size-=target.stat().st_size; target.unlink()
                    print("Rebuilding an invalid feature entry:",name[:16])

        if self.size>MAX_DISK_CACHE_GB*1024**3 or shutil.disk_usage(self.folder).free<150*1024**2:
            raise OSError("Feature cache limit reached. Save/version it, attach previous output and resume in fresh storage, or reduce the preset.")
        with op() as media:
            with (compressed(media) if codec else contextlib.nullcontext(media)) as path:
                features=self.encoder.extract(path,row.get("duration"))
        with atomic_path(target) as p:
            with p.open("wb") as f: np.savez_compressed(f,**features)
        self.size+=target.stat().st_size
        return features

print("[4/11] READY | Audio and face features", flush=True)

## 5 · Baselines ও delay-aware model



In [ ]:
TAPS=(-2,0,2)
def collect_labels(edges,spans):
    widths=np.diff(edges); target=np.zeros(len(widths),np.float32)
    for s,e in spans:
        overlap=np.maximum(0,np.minimum(edges[1:],e)-np.maximum(edges[:-1],s))
        target[overlap>=.5*np.minimum(widths,e-s)-1e-8]=1.
    return target

def crop_intervals(spans,lo,hi,offset=0.):
    return merge_spans([[max(lo,s+offset),min(hi,e+offset)] for s,e in spans if min(hi,e+offset)>max(lo,s+offset)])

def shifted(f,shift):
    n=len(f["ap"]); lo=max(0,shift); hi=min(n,n+shift)
    if hi<=lo: raise ValueError("Clip shorter than the requested shift.")
    return dict(A=f["A"][lo-shift:hi-shift],V=f["V"][lo:hi],ap=f["ap"][lo-shift:hi-shift],
                vp=f["vp"][lo:hi],edges=f["edges"][lo:hi+1],face=f["face"]),lo

def unit(x):
    return np.clip((x-x.mean())/max(1e-5,float(x.std())),-6,6).astype("f")

def alignment(f):
    # Estimate audio delay relative to video from envelope changes and lower-face motion.
    ap=unit(abs(np.diff(f["ap"],prepend=f["ap"][0]))); vp=unit(f["vp"])
    n=len(ap); m=min(5,(n-8)//2)
    if m<1 or ap.std()<1e-6 or vp.std()<1e-6: return 0,0.,ap,vp
    t=np.arange(m,n-m); v=unit(vp[t])
    corr={k:float(np.mean(unit(ap[t+k])*v)) for k in range(-m,m+1)}
    best=max(corr,key=lambda k:(corr[k],-abs(k)))
    confidence=max(0.,min(1.,corr[best]))
    if confidence<.25 or (best and corr[best]-corr[0]<.04): best=0
    return best,confidence,ap,vp

def _corr(a, b):
    """Pearson correlation, with constant windows counted as uninformative."""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    aa = a - a.mean()
    bb = b - b.mean()
    den = float(np.sqrt(np.dot(aa, aa) * np.dot(bb, bb)))
    if den < 1e-8:
        return 0.0
    return float(np.dot(aa, bb) / den)

def robust_alignment(f, window_bins=16, hop_bins=8, max_lag=5,
                     min_corr=.24, min_consensus=.50):
    """Estimate clip-wide lag using agreement across overlapping local windows.

    1. Derive audio energy changes and face-motion activity.
    2. Estimate the best cross-correlation lag in each local window.
    3. Exclude low-correlation windows (no reliable speech-motion match).
    4. Select the most frequently supported lag; reject low-consensus shifts.

    A positive returned lag means audio activity occurs later than mouth motion.
    The method does not inspect ground-truth annotations or model predictions.
    """
    ap = unit(np.abs(np.diff(f["ap"], prepend=f["ap"][0])))
    vp = unit(f["vp"])
    n = len(ap)
    m = min(int(max_lag), (n - 8) // 2)
    if m < 1 or ap.std() < 1e-6 or vp.std() < 1e-6:
        return 0, 0., ap, vp

    valid = np.arange(m, n - m)
    win = min(int(window_bins), len(valid))
    if win < 12:
        return 0, 0., ap, vp
    hop = max(1, int(hop_bins))
    starts = list(range(0, len(valid) - win + 1, hop))
    if starts[-1] != len(valid) - win:
        starts.append(len(valid) - win)

    votes = []
    lags = range(-m, m + 1)
    for start in starts:
        t = valid[start:start + win]
        v = vp[t]
        if float(np.std(v)) < .10:
            continue
        scores = {k: _corr(ap[t + k], v) for k in lags}
        candidate = max(scores, key=lambda k: (scores[k], -abs(k)))
        top = float(scores[candidate])
        if top >= min_corr:
            votes.append((int(candidate), top, scores[0]))

    if not votes:
        return 0, 0., ap, vp
    counts = {k: sum(1 for observed, _, _ in votes if observed == k) for k in lags}
    best = max(counts, key=lambda k: (counts[k], -abs(k)))
    supporters = [x for x in votes if x[0] == best]
    agreement = len(supporters) / len(votes)
    strength = float(np.median([x[1] for x in supporters]))
    confidence = max(0., min(1., agreement * max(0., strength)))
    if best != 0 and agreement < min_consensus:
        best = 0
    # Avoid declaring an offset when its advantage over no lag is marginal.
    if best != 0 and float(np.median([x[1] - x[2] for x in supporters])) < .04:
        best = 0
    return int(best), confidence, ap, vp

def design(f,kind):
    A,V=f["A"],f["V"]; n=len(A); t=np.arange(n)
    if kind=="audio": base=A
    elif kind=="visual": base=V
    else:
        lag,conf,ap,vp=(robust_alignment(f) if kind=="robust_delay" else alignment(f))
        ac=t+lag; vc=t-lag
        ia=ac.clip(0,n-1); iv=vc.clip(0,n-1)
        cross=np.column_stack([ap,vp,abs(ap-vp),abs(ap[ia]-vp),abs(ap-vp[iv]),
                               np.full(n,lag/5.),np.full(n,conf),ac==ia,vc==iv])
        if kind=="raw": base=np.column_stack([A,V,ap,vp,abs(ap-vp)])
        elif kind=="sync": base=cross
        elif kind in ("delay","robust_delay"):
            # Both native clocks remain present. Shifted evidence for matching is additional.
            base=np.column_stack([A,V,A[ia],V[iv],cross])
        else: raise ValueError(kind)
    return np.concatenate([base[(t+k).clip(0,n-1)] for k in TAPS],axis=1).astype("f")

def fit_windows(rows,cache,source,kind,channel,cap,augment=False):
    rng=np.random.default_rng(DATA_SEED+(0 if channel=="a" else 1))
    if len(rows)>cap:
        ids=np.sort(rng.choice(len(rows),size=cap,replace=False)); rows=[rows[i] for i in ids]
    allowance,extra=divmod(cap,len(rows)); X=[]; Xpair=[]; Y=[]
    for nr,r in enumerate(rows):
        f=cache.get(r,source); n=len(f["ap"]); y=collect_labels(f["edges"],r[channel]); sub=design(f,kind)
        limit=min(n,allowance+(nr<extra)); all_idx=np.arange(n); pair=None
        if augment and n>=8:
            mag=min(3,(n-4)//2); d=mag*(1 if int(uid(r["file"])[:4],16)%2 else -1)
            sf,lo=shifted(f,d); pair=design(sf,kind)
            # Match the same audio or visual content before and after a harmless shift.
            mapped=all_idx+(d if channel=="a" else 0)-lo
            all_idx=all_idx[(mapped>=0)&(mapped<len(pair))]
        limit=min(limit,len(all_idx))
        if not limit: continue
        pos=all_idx[y[all_idx]>0]
        want=min(len(pos),max(1,limit//2))
        ix=rng.choice(pos,size=want,replace=False) if want else np.empty(0,int)
        left=np.setdiff1d(all_idx,ix)
        if len(ix)<limit: ix=np.r_[ix,rng.choice(left,size=limit-len(ix),replace=False)]
        ix=np.sort(ix)
        X.append(sub[ix]); Y.append(y[ix])
        if augment: Xpair.append(pair[mapped[ix]] if pair is not None else sub[ix].copy())
    if not X: raise ValueError("No training windows.")
    X=np.concatenate(X); y=np.concatenate(Y)
    if len(np.unique(y))<2:
        raise ValueError(f"Only one window class for {channel}. Inspect metadata/timestamps and increase sample caps.")
    return X,y,(np.concatenate(Xpair) if augment else None)

def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z,-30,30)))

def score(head,X):
    z=np.clip((X-head["mean"])/head["scale"],-8,8)
    return sigmoid(np.maximum(z@head["W"]+head["b"],0)@head["out"]+head["bias"]).ravel()

def fit_head(X,y,pair,Xdev,ydev,seed,penalty):
    # Shared temporal filter over TAPS, ReLU, then a 1x1 binary head. All parameters trained.
    rng=np.random.default_rng(seed)
    mean=X.mean(0,dtype=np.float64).astype("f")
    scale=X.std(0,dtype=np.float64).astype("f"); scale[scale<1e-4]=1.
    X=np.clip((X-mean)/scale,-8,8)
    if pair is not None: pair=np.clip((pair-mean)/scale,-8,8)
    params={"W":(rng.standard_normal((X.shape[1],HIDDEN))*np.sqrt(2/X.shape[1])).astype("f"),
            "b":np.zeros(HIDDEN,"f"),"out":(rng.standard_normal((HIDDEN,1))/np.sqrt(HIDDEN)).astype("f"),
            "bias":np.zeros(1,"f")}
    m={k:np.zeros_like(v) for k,v in params.items()}; v={k:np.zeros_like(w) for k,w in params.items()}
    num=np.bincount(y.astype(int),minlength=2)
    weights=(len(y)/(2*num)).astype("f")
    best=-np.inf; winner=None; wait=0; steps=0; log=[]
    for epoch in range(1,EPOCHS+1):
        idx=rng.permutation(len(y))
        for j in range(0,len(idx),192):
            ids=idx[j:j+192]; z=X[ids]; labels=y[ids,None]; w=weights[y[ids].astype(int),None]
            h=np.maximum(z@params["W"]+params["b"],0); p=sigmoid(h@params["out"]+params["bias"])
            diff=w*(p-labels)/len(ids)
            if pair is None:
                pieces=[(z,h,diff)]
            else:
                z1=pair[ids]; h1=np.maximum(z1@params["W"]+params["b"],0)
                p1=sigmoid(h1@params["out"]+params["bias"])
                consistency=2*penalty*(p-p1)/len(ids)
                pieces=[(z,h,.5*diff+consistency*p*(1-p)),
                        (z1,h1,.5*w*(p1-labels)/len(ids)-consistency*p1*(1-p1))]
            grad={k:np.zeros_like(w0) for k,w0 in params.items()}
            for zz,hh,dd in pieces:
                dh=(dd@params["out"].T)*(hh>0)
                grad["W"]+=zz.T@dh; grad["b"]+=dh.sum(0)
                grad["out"]+=hh.T@dd; grad["bias"]+=dd.sum(0)
            steps+=1
            for k,g in grad.items():
                if k in ("W","out"): g+=1e-4*params[k]
                g=np.clip(g,-5,5)
                m[k]=.9*m[k]+.1*g; v[k]=.999*v[k]+.001*g*g
                params[k]-=.001*(m[k]/(1-.9**steps))/(np.sqrt(v[k]/(1-.999**steps))+1e-8)
        head=dict(mean=mean,scale=scale,**params)
        dev_ap=float(average_precision_score(ydev,score(head,Xdev)))
        log.append({"epoch":epoch,"dev_window_AP":dev_ap})
        if dev_ap>best+1e-6:
            best=dev_ap; winner={k:a.copy() for k,a in head.items()}; wait=0
        else:
            wait+=1
            if wait>=4: break
    return winner,log

# Each fusion returns two predictions on the original displayed timeline.
MODELS={"audio_only":{"a":"audio_a"},"visual_only":{"v":"visual_v"},
        "late_fusion":{"a":"audio_a","v":"visual_v"},
        "raw_fusion":{"a":"raw_a","v":"raw_v"},
        "alignment_only":{"a":"sync_a","v":"sync_v"},
        "delay_aware":{"a":"delay_a","v":"delay_v"},
        "robust_delay_aware":{"a":"robust_a","v":"robust_v"}}
if USE_SHIFT_CONSISTENCY:
    MODELS["delay_without_consistency"]={"a":"delay0_a","v":"delay0_v"}

def trained_heads(tr,dev,cache,src,seed,store_path=None,previous_path=None):
    heads={}; logs=[]
    for name in sorted({h for d in MODELS.values() for h in d.values()}):
        typ,ch=name.split("_"); kind="robust_delay" if typ=="robust" else ("delay" if typ=="delay0" else typ)
        local=(store_path/(name+".npz")) if store_path else None
        prior=(previous_path/(name+".npz")) if previous_path else None
        complete=next((path for path in (local,prior) if path and path.is_file()),None)
        if complete:
            with np.load(complete,allow_pickle=False) as data:
                heads[name]={k:data[k] for k in ("mean","scale","W","b","out","bias")}
                logs.extend(json.loads(data["history"].item()))
            if local and complete!=local:
                with atomic_path(local) as temp: shutil.copyfile(complete,temp)
            print("  Resumed head:",name)
            continue
        print(f"  Fitting {name} ...", flush=True)
        X,y,xpair=fit_windows(tr,cache,src,kind,ch,MAX_FIT_WINDOWS,augment=USE_SHIFT_CONSISTENCY)
        xd,yd,_=fit_windows(dev,cache,src,kind,ch,min(10000,MAX_FIT_WINDOWS),augment=False)
        head,history=fit_head(X,y,xpair,xd,yd,seed+(0 if ch=="a" else 1),
                              penalty=.5 if typ in ("delay","robust") and USE_SHIFT_CONSISTENCY else 0.)
        heads[name]=head
        history=[dict(seed=seed,head=name,**rec) for rec in history]
        logs.extend(history)
        if local:
            with atomic_path(local) as temp:
                with temp.open("wb") as ff: np.savez_compressed(ff,**head,history=json.dumps(history))
        print(f"  {name}: {len(y)} train windows; best dev window AP {max(t['dev_window_AP'] for t in history):.3f}")
    return heads,logs

def head_scores(f,heads):
    mats={}; output={}
    for name,h in heads.items():
        typ=name.split("_")[0]; typ="robust_delay" if typ=="robust" else ("delay" if typ=="delay0" else typ)
        if typ not in mats: mats[typ]=design(f,typ)
        output[name]=score(h,mats[typ])
    return output

print("[5/11] READY | Temporal model definitions", flush=True)

## 6· Ground truth  prediction false alarm



In [ ]:
def tiou(x,y):
    overlap=max(0.,min(x[1],y[1])-max(x[0],y[0]))
    union=(x[1]-x[0])+(y[1]-y[0])-overlap
    return overlap/union if union>0 else 0.

def runs(p,edges,threshold):
    idx=np.diff(np.r_[False,np.asarray(p)>=threshold,False].astype(int))
    return [(float(edges[i]),float(edges[j]),float(.5*p[i:j].mean()+.5*p[i:j].max()))
            for i,j in zip(np.flatnonzero(idx==1),np.flatnonzero(idx==-1))]

def proposals(p,edges):
    candidates=[]
    for threshold in (.10,.20,.35,.50,.65,.80,.90):
        candidates.extend(runs(p,edges,threshold))
    kept=[]
    for row in sorted(candidates,key=lambda q:(-q[2],q[0],q[1])):
        if all(tiou(row,k)<.65 for k in kept):
            kept.append(row)
            if len(kept)==100: break
    return kept

def greedy(proposed,truth,cut):
    used=set(); result=[]
    for p in proposed:
        matches=[(tiou(p,g),j) for j,g in enumerate(truth) if j not in used]
        best=max(matches,default=(0.,-1))
        yes=best[0]>=cut
        if yes: used.add(best[1])
        result.append(int(yes))
    return result

def t_counts(pred,truth,cut=.5):
    p=sorted(pred,key=lambda q:-q[2]); tp=sum(greedy(p,truth,cut))
    return tp,len(p)-tp,len(truth)-tp

def div(n,d): return float(n/d) if d else float("nan")

def f1(tp,fp,fn): return div(2*tp,2*tp+fp+fn)

def temporal_ap(items,cut):
    # Continuous all-points interpolated AP. Rank across clips, match once within a clip.
    ranked=[];total=0
    for key,gt,pred in items:
        if gt is None: continue
        total+=len(gt); ordered=sorted(pred,key=lambda p:(-p[2],p[0],p[1]))
        for p,hit in zip(ordered,greedy(ordered,gt,cut)): ranked.append((p[2],hit))
    if not total: return np.nan
    if not ranked: return 0.
    ranked.sort(key=lambda x:-x[0])
    sc=np.array([x[0] for x in ranked]); tp=np.cumsum([x[1] for x in ranked])
    ends=np.r_[np.flatnonzero(sc[:-1]!=sc[1:]),len(sc)-1]
    prec=tp[ends]/(ends+1); rec=tp[ends]/total
    prec=np.maximum.accumulate(prec[::-1])[::-1]
    return float(np.sum(np.diff(np.r_[0.,rec])*prec))

def tuned(dev_records,head,channel):
    best=None
    for th in np.arange(.10,.951,.05):
        tp=fp=fn=0
        for r,f,scores in dev_records:
            x=t_counts(runs(scores[head],f["edges"],th),r[channel])
            tp+=x[0];fp+=x[1];fn+=x[2]
        val=f1(tp,fp,fn)
        if not np.isfinite(val): continue
        candidate=(val,-fp,float(th))
        if best is None or candidate>best: best=candidate
    if best is None: raise ValueError("Cannot choose a threshold without positive dev intervals.")
    return best[2]

def threshold_all(records,heads):
    return {n:tuned(records,n,n.split("_")[-1]) for n in heads}

def small_f(f):
    lag,confidence,_,_=alignment(f)
    rlag,rconfidence,_,_=robust_alignment(f)
    return {"edges":f["edges"],"face":float(f["face"]),"lag":lag*STEP,
            "alignment_confidence":confidence,"robust_lag":rlag*STEP,
            "robust_alignment_confidence":rconfidence}

def on_reference_grid(f,reference):
    # H.264/AAC can round the final frame. Put compressed features back on the clean media clock.
    e,ref=f["edges"],reference["edges"]
    if max(abs(e[0]-ref[0]),abs(e[-1]-ref[-1]))>STEP+.01:
        raise ValueError("Re-encoding changed media timing; review the clip before comparing conditions.")
    if np.array_equal(e,ref): return f
    centers=(e[1:]+e[:-1])/2; new=(ref[1:]+ref[:-1])/2
    out=dict(f,edges=ref)
    for k in ("A","V"): out[k]=np.column_stack([np.interp(new,centers,col) for col in f[k].T]).astype("f")
    for k in ("ap","vp"): out[k]=np.interp(new,centers,f[k]).astype("f")
    return out

def record_predictions(rows,cache,src,heads,shift=0,codec=False,matched=False):
    records=[]
    for original in rows:
        try:
            ref=cache.get(original,src)
            target=on_reference_grid(cache.get(original,src,True),ref) if codec else ref
            r=dict(original,original_label=original.get("label"))
            if matched:
                # Fixed central support for every stress condition, independent of label and lag sign.
                edges=ref["edges"]; margin=3
                if len(edges)-1<=2*margin+1: continue
                ix=np.arange(margin,len(edges)-1-margin)
                f=dict(edges=edges[margin:len(edges)-margin],face=target["face"],
                       A=target["A"][ix-shift],ap=target["ap"][ix-shift],
                       V=target["V"][ix],vp=target["vp"][ix])
                lo,hi=f["edges"][[0,-1]]
                for c,sec in (("a",shift*STEP),("v",0.)):
                    if r[c] is not None: r[c]=crop_intervals(r[c],lo,hi,sec)
                if r["a"] is not None and r["v"] is not None: r["label"]=int(bool(r["a"] or r["v"]))
            else:
                if shift or codec: raise ValueError("Stress evaluations must use matched support.")
                f=target
            records.append((r,small_f(f),head_scores(f,heads)))
        except Exception as e:
            raise RuntimeError(f"Failed for {original['file']}: {e}") from e
    if not records: raise ValueError("No eligible clips remain on the evaluation support.")
    return records

def describe(r,f,scores,model,model_set,thresholds):
    by_channel={}; ranks={}
    for c,n in model_set[model].items():
        by_channel[c]=runs(scores[n],f["edges"],thresholds[n])
        ranks[c]=proposals(scores[n],f["edges"])
    active=list(model_set[model].values())
    combined=np.maximum.reduce([scores[n] for n in active])
    union=merge_spans([p[:2] for v in by_channel.values() for p in v])
    any_segments=[]
    for s,e in union:
        mask=(f["edges"][:-1]<e)&(f["edges"][1:]>s)
        any_segments.append((s,e,float(combined[mask].max())))
    by_channel["any"]=any_segments; ranks["any"]=proposals(combined,f["edges"])
    gt={"a":r.get("a"),"v":r.get("v")}
    gt["any"]=merge_spans(gt["a"]+gt["v"]) if all(x is not None for x in (gt["a"],gt["v"])) else None
    return {"file":r["file"],"group":r.get("group") or r["file"],"speaker":r.get("speaker",""),
            "label":r.get("label"),"original_label":r.get("original_label",r.get("label")),"kind":r.get("kind","external"),"gt":gt,
            "group_verified":r.get("group_verified",False),
            "pred":by_channel,"ranked":ranks,"clip_score":float(combined.max()),
            "alarm":bool(union),"estimated_audio_lag_s":float(f["robust_lag"] if model=="robust_delay_aware" else f["lag"]),"alignment_confidence":float(f["robust_alignment_confidence"] if model=="robust_delay_aware" else f["alignment_confidence"]),"face_tracking_fraction":float(f["face"]),
            "evaluated_time_s":[float(f["edges"][0]),float(f["edges"][-1])]}

def report(dets):
    y=[d["label"] for d in dets if d["label"] is not None]
    known=[d for d in dets if d["label"] is not None]
    real=[d for d in known if d["original_label"]==0]; alarms=sum(d["alarm"] for d in real)
    pos=[d for d in known if d["label"]==1]; hits=sum(d["alarm"] for d in pos)
    out={"clips":len(dets),"clip_labeled":len(known),"genuine_clips":len(real),"genuine_false_alarms":alarms,
         "genuine_FPR":div(alarms,len(real)),"clip_F1":f1(hits,sum(d["alarm"] for d in known if d["label"]==0),len(pos)-hits) if pos else np.nan,
         "clip_AUROC":float(roc_auc_score(y,[d["clip_score"] for d in known])) if len(set(y))==2 else np.nan,
         "unlabeled_clips":len(dets)-len(known),"all_clip_alert_rate":div(sum(d["alarm"] for d in dets),len(dets))}
    for track in ("a","v","any"):
        items=[(d["file"],d["gt"][track],d["ranked"][track]) for d in dets if track in d["pred"] and d["gt"][track] is not None]
        counts=[t_counts(d["pred"][track],d["gt"][track]) for d in dets if track in d["pred"] and d["gt"][track] is not None]
        tp,fp,fn=np.asarray(counts,int).reshape(-1,3).sum(0)
        out.update({track+"_annotated_clips":len(items),track+"_GT_segments":int(tp+fn),track+"_TP":int(tp),
                    track+"_FP":int(fp),track+"_FN":int(fn),track+"_segment_F1@50":f1(tp,fp,fn) if tp+fn else np.nan})
        for cut in (.5,.75):
            out[f"{track}_AP@{round(100*cut)}"]=temporal_ap(items,cut)
    for k in (50,75):
        ps=[out[f"{c}_AP@{k}"] for c in ("a","v")]
        out[f"AV_macro_mAP@{k}"]=float(np.mean(ps)) if np.isfinite(ps).all() else np.nan
    # Cross-modal attribution errors are hidden by any-stream interval metrics.
    visual_only=[d for d in dets if d["kind"]=="visual" and "a" in d["pred"]]
    audio_only=[d for d in dets if d["kind"]=="audio" and "v" in d["pred"]]
    out["audio_alarm_on_visual_only_clips"]=sum(bool(d["pred"]["a"]) for d in visual_only)
    out["visual_only_clips_with_audio_head"]=len(visual_only)
    out["visual_alarm_on_audio_only_clips"]=sum(bool(d["pred"]["v"]) for d in audio_only)
    out["audio_only_clips_with_visual_head"]=len(audio_only)
    out["audio_spurious_on_visual_rate"]=div(out["audio_alarm_on_visual_only_clips"],len(visual_only))
    out["visual_spurious_on_audio_rate"]=div(out["visual_alarm_on_audio_only_clips"],len(audio_only))
    return out

def cluster_ci(dets,seed=903,nrep=400):
    # Resample entire speakers when all IDs are known, otherwise entire available source groups.
    level="speaker" if all(d["speaker"] for d in dets) else "group"
    if level=="group" and not all(d["group_verified"] for d in dets):
        return {n:dict(lo=None,hi=None,valid_replicates=0,clusters=None,group_type="IDs incomplete") for n in ("genuine_FPR","any_segment_F1@50")}
    clusters=defaultdict(lambda:np.zeros(5))
    for d in dets:
        c=clusters[d[level]]
        if d["original_label"]==0: c[:2]+=[int(d["alarm"]),1]
        if d["gt"]["any"] is not None: c[2:]+=t_counts(d["pred"]["any"],d["gt"]["any"])
    vals=np.array(list(clusters.values())); rng=np.random.default_rng(seed); estimates=[[],[]]
    if len(vals)>=2:
        for _ in range(nrep):
            fp,n,tp,sfp,fn=vals[rng.integers(len(vals),size=len(vals))].sum(0)
            if n: estimates[0].append(fp/n)
            if tp+fn: estimates[1].append(f1(tp,sfp,fn))
    result={}
    for name,sample in zip(("genuine_FPR","any_segment_F1@50"),estimates):
        result[name]={"lo":float(np.quantile(sample,.025)) if sample else None,"hi":float(np.quantile(sample,.975)) if sample else None,
                      "valid_replicates":len(sample),"clusters":len(vals),"group_type":level}
    return result

print("[6/11] READY | Localization metrics", flush=True)

## 7 · Model, prediction ,Figure


In [ ]:
def save_bundle(path,heads,meta):
    with atomic_path(path) as temp:
        with zipfile.ZipFile(temp,"w",compression=zipfile.ZIP_DEFLATED) as z:
            z.writestr("bundle.json",json.dumps(json_ready(meta),ensure_ascii=False,allow_nan=False))
            for name,head in heads.items():
                b=io.BytesIO(); np.savez(b,**head); z.writestr("heads/"+name+".npz",b.getvalue())

def open_bundle(path):
    with zipfile.ZipFile(path) as z:
        info=json.loads(z.read("bundle.json"))
        if info.get("code")!=CODE_VERSION or info["step"]!=STEP:
            raise ValueError("This checkpoint uses a different code version/time grid.")
        names=set(v for heads in info["models"].values() for v in heads.values())
        params={}
        for name in sorted(names):
            if not re.fullmatch(r"[a-z0-9_]+",name): raise ValueError("Unexpected head name.")
            with np.load(io.BytesIO(z.read("heads/"+name+".npz")),allow_pickle=False) as a:
                params[name]={k:a[k] for k in ("mean","scale","W","b","out","bias")}
    return params,info

def enforce_spec(encoder,meta):
    # Different ffmpeg package versions are logged. Feature definitions and weights must match.
    keys=set(encoder.spec)|set(meta["encoder"])
    mismatch=[k for k in keys if k!="ffmpeg" and encoder.spec.get(k)!=meta["encoder"].get(k)]
    if mismatch: raise ValueError("Checkpoint/extractor mismatch: "+str(mismatch)+". Mount the same weights/runtime.")

def predict_clip(video,encoder,heads,meta,model=None):
    model=model or EVAL_MODEL
    if model not in meta["models"]: raise KeyError(model)
    enforce_spec(encoder,meta)
    f=encoder.extract(Path(video))
    limited={v:heads[v] for v in meta["models"][model].values()}
    s=head_scores(f,limited)
    d=describe({"file":Path(video).name,"label":None},small_f(f),s,model,meta["models"],meta["thresholds"])
    return {"file":d["file"],"model":model,"evaluated_time_s":d["evaluated_time_s"],
            "clip_score":d["clip_score"],"flagged":d["alarm"],
            "global_audio_delay_s_estimate":d["estimated_audio_lag_s"],
            "audio":[dict(start_s=x,end_s=y,score=p) for x,y,p in d["pred"].get("a",[])],
            "visual":[dict(start_s=x,end_s=y,score=p) for x,y,p in d["pred"].get("v",[])]},(f,s)

def timeline(path,r,f,scores,model,meta):
    import matplotlib.pyplot as plt
    chans=meta["models"][model]
    fig,axes=plt.subplots(len(chans),1,figsize=(9,2.25*len(chans)),sharex=True,squeeze=False)
    for ax,(ch,name) in zip(axes[:,0],chans.items()):
        e=f["edges"]; ax.stairs(scores[name],e,color="#036b68",label="model score")
        ax.axhline(meta["thresholds"][name],color="#8e4320",linestyle="--",label="dev threshold")
        for i,(left,right) in enumerate(r.get(ch) or []):
            ax.axvspan(left,right,color="#f0b34b",alpha=.3,label="annotated edit" if i==0 else None)
        ax.set(ylim=(-.02,1.02),ylabel="Audio" if ch=="a" else "Visual"); ax.legend(loc="upper right",fontsize=7)
    axes[-1,0].set_xlabel("Time from original media origin (s)")
    fig.suptitle(model+" | "+str(r["file"]).split("/")[-1]); fig.tight_layout()
    fig.savefig(path,dpi=150,bbox_inches="tight"); plt.close(fig)

def save_eval(directory,records,model_map,thresholds,split,condition,seed,keep_curves=True):
    directory.mkdir(parents=True,exist_ok=True)
    summaries=[]; intervals=[]; pairs={}; subgroup=[]
    with atomic_path(directory/"predictions.jsonl") as tmp:
        with tmp.open("w",encoding="utf-8") as stream:
            for model in model_map:
                detections=[describe(r,f,s,model,model_map,thresholds) for r,f,s in records]
                summaries.append(dict(seed=seed,split=split,condition=condition,model=model,**report(detections)))
                if model in ("late_fusion","delay_aware","raw_fusion","robust_delay_aware"):
                    pairs[model]={d["file"]:(d["speaker"],d["group"],d["group_verified"],per_clip_counts(d)) for d in detections}
                for kind in sorted(set(d["kind"] for d in detections)):
                    selected=[d for d in detections if d["kind"]==kind]
                    subgroup.append(dict(seed=seed,split=split,condition=condition,model=model,clip_kind=kind,
                                         clips=len(selected),flagged_clips=sum(d["alarm"] for d in selected),
                                         labeled_positive=sum(d["label"]==1 for d in selected)))
                cis=cluster_ci(detections)
                for metric,value in cis.items(): intervals.append(dict(seed=seed,split=split,condition=condition,model=model,metric=metric,**value))
                for d in detections:
                    item={k:v for k,v in d.items() if k!="ranked"}
                    if keep_curves: item["ranked"]=d["ranked"]
                    stream.write(json.dumps(json_ready(dict(model=model,**item)),ensure_ascii=False,allow_nan=False)+"\n")
    write_csv(directory/"metrics.csv",summaries); write_csv(directory/"cluster_ci.csv",intervals)
    write_csv(directory/"paired_difference_ci.csv",paired_gaps(pairs))
    write_csv(directory/"per_manipulation.csv",subgroup)
    return summaries

def summarize_seeds(path,rows):
    if not rows: return
    df=pd.DataFrame(rows)
    ids=["split","condition","model"]
    vals=["any_AP@50","any_AP@75","AV_macro_mAP@50","any_segment_F1@50","genuine_FPR","clip_AUROC"]
    summary=df.groupby(ids,dropna=False)[vals].agg(["mean","std","count"])
    summary.columns=["_".join(c) for c in summary.columns];write_csv(path,summary.reset_index())

def official_compare(path,rows,method=None):
    # Authors' post_process.py writes file -> [[confidence, BEGIN_FRAME, END_FRAME], ...].
    # metrics.py divides frame bounds by 25. Clip-level-only modalities cannot be reconstructed.
    raw=json.loads(Path(path).read_text()); raw={safe_key(k):v for k,v in raw.items()}
    items=[]
    for r in rows:
        if r["file"] not in raw: raise ValueError("BA-TFD output missing selected test file: "+r["file"])
        props=[]
        for confidence,s,e in raw[r["file"]]:
            if not np.isfinite([confidence,s,e]).all() or not 0<=s<e:
                raise ValueError("Invalid BA-TFD proposals.")
            props.append((s/25.,e/25.,float(confidence)))
        items.append((r["file"],merge_spans(r["a"]+r["v"]),props))
    return {"model":"official_"+(method or BA_TFD_LABEL)+"_fusion","clips":len(items),"any_AP@50":temporal_ap(items,.5),
            "any_AP@75":temporal_ap(items,.75),"protocol":"same selected IDs; shared AP implementation; native official proposals"}

print("[7/11] READY | Checkpoint and prediction tools", flush=True)

## 8 · Official baseline And paired comparison



In [ ]:
def run_official(source,test_rows,dest):
    """Run the authors' own scripts in the user's separate compatible environment."""
    repo=Path(OFFICIAL_REPO).resolve(); ck=Path(OFFICIAL_CKPT).resolve()
    py=str(Path(OFFICIAL_PYTHON).resolve()) if OFFICIAL_PYTHON else sys.executable
    cfg=(repo/OFFICIAL_CONFIG).resolve()
    if not (repo/"evaluate.py").is_file() or not ck.is_file() or not cfg.is_file():
        raise FileNotFoundError("Mount the official repository, desired .toml and its matching .ckpt.")
    import tomllib
    with cfg.open("rb") as f: name=tomllib.load(f)["name"]
    dest.mkdir(exist_ok=True,parents=True)
    with tempfile.TemporaryDirectory(prefix="lav-official-") as tmp:
        work=Path(tmp); data=work/"data"; data.mkdir()
        meta=[]; total_bytes=0
        for r in test_rows:
            loc=source.locate(r["file"]); dst=data/safe_key(r["file"]); dst.parent.mkdir(exist_ok=True,parents=True)
            if not all(r[k] is not None for k in ("video_frames","audio_frames","audio_channels")):
                raise ValueError("Official baseline needs the release's video_frames, audio_frames and audio_channels.")
            if source.kind=="folder": dst.symlink_to(loc)
            else:
                with source.media(loc) as video:
                    if shutil.disk_usage(work).free < video.stat().st_size+256*1024**2:
                        raise OSError("Insufficient scratch space for official evaluator/test subset.")
                    shutil.copyfile(video,dst); total_bytes+=dst.stat().st_size
            meta.append(dict(file=r["file"],n_fakes=r["n_fakes"],fake_periods=merge_spans(r["a"]+r["v"]),
                             duration=r["duration"],original=r["original"],split="test",
                             modify_video=bool(r["v"]),modify_audio=bool(r["a"]),video_frames=r["video_frames"],
                             audio_channels=r["audio_channels"],audio_frames=r["audio_frames"]))
        write_json(data/"metadata.min.json",meta)
        # Its released Lightning stack and native dependencies belong in a separate matched environment.
        args=[py,str(repo/"evaluate.py"),"--config",str(cfg),"--data_root",str(data),"--checkpoint",str(ck),
              "--batch_size","1","--num_workers","1","--modalities","fusion","--gpus","1"]
        with (dest/"official_run.log").open("w") as log:
            status=subprocess.run(args,cwd=work,stdout=log,stderr=subprocess.STDOUT)
        if status.returncode: raise RuntimeError(f"Official evaluator exited {status.returncode}. See {dest/'official_run.log'}; you may instead supply its exported proposal JSON.")
        made=work/"output"/"results"/(name+".json")
        if not made.is_file(): raise FileNotFoundError(made)
        with atomic_path(dest/"official_fusion.json") as t: shutil.copyfile(made,t)
        write_json(dest/"provenance.json",{"config_file":cfg.name,"config_sha256":digest_file(cfg),"checkpoint_sha256":digest_file(ck),
                  "official_evaluate_py_sha256":digest_file(repo/"evaluate.py"),"test_ids":len(test_rows),"proposal_units":"frames at 25 fps"})
    return dest/"official_fusion.json"

def per_clip_counts(d):
    if d["gt"]["any"] is not None: seg=t_counts(d["pred"]["any"],d["gt"]["any"])
    else: seg=(0,0,0)
    return np.array([int(d["alarm"] and d["original_label"]==0),int(d["original_label"]==0),*seg],dtype=float)

def paired_gaps(summaries,nrep=400):
    """Paired differences in FPR and segment F1, including the new LWCA model."""
    comparisons=(("delay_aware","late_fusion"),
                 ("robust_delay_aware","delay_aware"),
                 ("robust_delay_aware","raw_fusion"),
                 ("robust_delay_aware","late_fusion"))
    out=[]
    for proposed,reference in comparisons:
        if proposed not in summaries or reference not in summaries: continue
        left,right=summaries[proposed],summaries[reference]
        if left.keys()!=right.keys() or not left: raise ValueError("Paired comparison clip IDs differ")
        contrast=proposed+" minus "+reference
        if all(left[k][0] for k in left): level=0; group_type="speaker"
        elif all(left[k][2] for k in left): level=1; group_type="source"
        else:
            out.extend(dict(metric=m,delta=None,lo=None,hi=None,cluster_type="IDs incomplete",
                            contrast=contrast) for m in ("genuine_FPR","any_segment_F1@50"))
            continue
        groups=defaultdict(lambda:np.zeros((2,5)))
        for k in left:
            groups[left[k][level]][0]+=left[k][3]
            groups[left[k][level]][1]+=right[k][3]
        a=np.array(list(groups.values()))
        rng=np.random.default_rng(601)
        for metric in ("genuine_FPR","any_segment_F1@50"):
            def difference(counts):
                vals=[]
                for fp,n,tp,sfp,fn in counts:
                    vals.append(div(fp,n) if metric=="genuine_FPR" else (f1(tp,sfp,fn) if tp+fn else np.nan))
                return vals[0]-vals[1]
            full=difference(a.sum(0)); boot=[]
            if len(a)>=2:
                for _ in range(nrep):
                    x=difference(a[rng.integers(len(a),size=len(a))].sum(0))
                    if np.isfinite(x): boot.append(x)
            out.append(dict(metric=metric,delta=full,
                            lo=float(np.quantile(boot,.025)) if boot else None,
                            hi=float(np.quantile(boot,.975)) if boot else None,
                            clusters=len(a),cluster_type=group_type,
                            valid_replicates=len(boot),contrast=contrast))
    return out

print("[8/11] READY | Baseline comparison", flush=True)

## 9 · Bangla external test


In [ ]:
EXTERNAL_FIELDS=["dataset","language","clip_id","path","speaker_id","source_id","clip_label",
                 "audio_segments","visual_segments","generator_family"]
def draft_external_csv(root,name,path,language=""):
    """Make an unverified template; subfolders and filenames never imply class or language."""
    root=Path(root).resolve(); path=Path(path)
    if path.exists(): raise FileExistsError(f"Protecting an existing manifest: {path}")
    files=sorted(x for x in root.rglob("*") if x.suffix.lower() in {".mp4",".mkv",".mov",".avi"} and x.is_file())
    if not files: raise FileNotFoundError("No videos found under "+str(root))
    rows=[dict(dataset=name,language=language,clip_id=str(x.relative_to(root)),path=str(x),speaker_id="",
               source_id="",clip_label="unknown",audio_segments="",visual_segments="",generator_family="") for x in files]
    write_csv(path,rows)
template=OUT/"external_manifest_template.csv"
if not template.exists(): pd.DataFrame(columns=EXTERNAL_FIELDS).to_csv(template,index=False)
for dataset,root,language in (("InDeepFake",INDEEPFAKE_VIDEO_ROOT,""),("BanglaLipToText",LIP_TO_TEXT_VIDEO_ROOT,"bn")):
    draft=OUT/(dataset+"_draft.csv")
    if root and not draft.exists(): draft_external_csv(root,dataset,draft,language)

def get_external(path,base,cache):
    root=Path(base).resolve() if base else Path(path).resolve().parent
    seen=set(); absolute=set(); rows=[]
    for index,line in enumerate(read_rows(path),2):
        try:
            if not set(EXTERNAL_FIELDS).issubset(line): raise ValueError("Use external_manifest_template.csv headers.")
            d={k:(line[k] or "").strip() for k in EXTERNAL_FIELDS}
            if not d["dataset"] or not d["language"] or not d["clip_id"]: raise ValueError("dataset, verified language, and clip_id are required.")
            key=(d["dataset"],d["clip_id"])
            if key in seen: raise ValueError("Duplicate external clip_id")
            seen.add(key)
            p=(root/d["path"]).resolve()
            if not p.is_file(): raise FileNotFoundError(p)
            if p in absolute: raise ValueError("Same external video path listed twice.")
            absolute.add(p)
            lab={"real":0,"fake":1,"unknown":None,"":None}.get(d["clip_label"].lower(),"invalid")
            if lab=="invalid": raise ValueError("clip_label must be real, fake, or unknown.")
            # First extract independently of any label, getting the actual recorded common time.
            row=dict(file=d["clip_id"],path=str(p),dataset=d["dataset"],language=d["language"],
                     speaker=d["speaker_id"],group=d["source_id"] or d["clip_id"],group_verified=bool(d["source_id"]),
                     label=lab,kind="external",family=d["generator_family"])
            f=cache.get(row); dur=float(f["edges"][-1])
            for col,ch in (("audio_segments","a"),("visual_segments","v")):
                val=d[col]
                row[ch]=checked_spans(json.loads(val),dur) if val else ([] if lab==0 else None)
            if lab==0 and (row["a"] or row["v"]): raise ValueError("Verified genuine clip has a nonempty edit annotation.")
            complete=row["a"] is not None and row["v"] is not None
            if complete:
                expected=int(bool(row["a"] or row["v"]))
                if lab is not None and lab!=expected: raise ValueError("Whole-clip and interval labels disagree.")
                row["label"]=expected
            if lab is None and (row["a"] or row["v"]): row["label"]=1
            rows.append(row)
        except Exception as err:
            raise ValueError(f"External manifest row {index}: {err}") from err
    if not rows: raise ValueError("Populate the external manifest before enabling external testing.")
    return rows

def evaluate_external(manifest,encoder,cache,heads,meta,dest,lav_groups=(),lav_speakers=()):
    rows=get_external(manifest,EXTERNAL_ROOT,cache)
    if set(r["group"] for r in rows if r["group_verified"]) & set(lav_groups):
        raise ValueError("An explicitly mapped external source is also present in development data.")
    if set(r["speaker"] for r in rows if r["speaker"]) & set(lav_speakers):
        raise ValueError("An explicitly mapped external speaker is also present in development data.")
    train_sources=set(meta.get("development_source_hashes",[])); train_speakers=set(meta.get("development_speaker_hashes",[]))
    if any((r["group_verified"] and uid(["source",r["group"]]) in train_sources) or
           (r["speaker"] and uid(["speaker",r["speaker"]]) in train_speakers) for r in rows):
        raise ValueError("A mapped source/speaker overlaps the checkpoint development set.")
    train_fams={s.strip().casefold() for s in TRAIN_GENERATOR_FAMILIES if s.strip()}
    hold={s.strip().casefold() for s in HELDOUT_EXTERNAL_FAMILIES if s.strip()}
    if hold and (not train_fams or train_fams & hold):
        raise ValueError("Document training generators and choose disjoint external families.")
    groups=defaultdict(list)
    for r in rows: groups[(r["dataset"],r["language"])].append(r)
    all_summary=[]
    for (dataset,lang),subset in groups.items():
        d=dest/(tokensafe(dataset)+"_"+tokensafe(lang)+"_"+uid([dataset,lang])[:6])
        rec=record_predictions(subset,cache,None,heads)
        metrics=save_eval(d,rec,meta["models"],meta["thresholds"],"external:"+dataset+":"+lang,"clean",meta["seed"])
        all_summary.extend(metrics)
        if hold:
            def relevant(row):
                fams={s.strip().casefold() for s in row["family"].split("|") if s.strip()}
                return row["label"]==1 and bool(fams & hold) and not (fams & train_fams)
            unseen=[e for e in rec if e[0]["label"]==0 or relevant(e[0])]
            if any(relevant(r) for r,_,_ in unseen) and any(r["label"]==0 for r,_,_ in unseen):
                all_summary.extend(save_eval(d/"documented_unseen_family",unseen,meta["models"],meta["thresholds"],
                                   "external_unseen:"+dataset+":"+lang,"clean",meta["seed"]))
            write_json(d/"declared_generator_protocol.json",{"train_families":sorted(train_fams),"heldout_families":sorted(hold),
                       "eligible_fakes":sum(relevant(r) for r in subset),"genuine_controls":sum(r["label"]==0 for r in subset)})
        write_json(d/"annotation_coverage.json",{"n":len(subset),"clip_labels":sum(r["label"] is not None for r in subset),
                 "audio_intervals_known":sum(r["a"] is not None for r in subset),"visual_intervals_known":sum(r["v"] is not None for r in subset),
                 "source_ids_supplied":sum(r["group_verified"] for r in subset),"speaker_ids_supplied":sum(bool(r["speaker"]) for r in subset),
                 "generator_families_reported":dict(Counter(r["family"] for r in subset if r["family"]))})
    write_csv(dest/"external_metrics.csv",all_summary)
    return all_summary

print("[9/11] READY | External evaluation tools", flush=True)

## 10 · training Start

In [ ]:
print("[10/11] RUN | " + ACTION, flush=True)

def warm(rows, cache, src):
    total = len(rows)
    start = last = monotonic()
    for i, row in enumerate(rows, 1):
        if i == 1:
            print(f"  Reading clip 1/{total}: {row['file']}", flush=True)
        try:
            cache.get(row, src)
        except Exception as err:
            raise RuntimeError(f"{row['file']}: {err}") from err
        now = monotonic()
        if i == 1 or i == total or i % 25 == 0 or now-last >= 15:
            print(f"  Features {i}/{total} ({now-start:.0f}s elapsed)", flush=True)
            last = now

def train_project():
    print("Opening LAV-DF metadata ...", flush=True)
    src=LavSource(DATA_PATH or auto_dataset())
    try:
        print("Reading metadata and checking labels and split IDs ...", flush=True)
        base, rejected = lav_rows(src)
        original_count = len(base) + len(rejected)
        if any(r["self_check"] for r in base) and not all(r["self_check"] for r in base):
            raise ValueError("Keep generated self-check clips separate from research data.")
        self_check=all(r["self_check"] for r in base)
        if self_check: print("SELF-CHECK. These generated clips and resulting scores are for software verification only.")
        base, reserved, holdout = reserve_previous_test(base)
        # Quarantine any surviving row sharing a file ID with a rejected metadata version.
        disputed = {r["file"] for r in rejected if r.get("file")}
        if disputed:
            clean_rows = []
            for r in base:
                if r["file"] in disputed:
                    rejected.append(dict(file=r["file"], split=r["split"], reason="ID shared with a quarantined row"))
                else:
                    clean_rows.append(r)
            base = clean_rows
        if RUN_FINAL_TEST and not any(r["split"] == "test" for r in base):
            raise ValueError("No unexposed LAV-DF test clips remain. Attach the full release and recheck the audit.")
        if ALLOW_PARTIAL_INPUT:
            available_rows=[]
            for row in base:
                try: src.locate(row["file"])
                except FileNotFoundError: continue
                available_rows.append(row)
            base=available_rows
            required={"train","dev","test"} if RUN_FINAL_TEST else {"train","dev"}
            if not required.issubset({r["split"] for r in base}): raise ValueError("Supplied media misses a required official split.")
        caps={"quick":(64,32,32),"pilot":(320,96,160),"study":(3200,800,1200),"full":(None,None,None)}[PRESET]
        selected={s:picked(base,s,n) for s,n in zip(("train","dev","test"),caps)}
        # IDs are fixed before any feature or model is fitted. Test features are read only in a final run.
        used=selected["train"]+selected["dev"]+(selected["test"] if RUN_FINAL_TEST else [])
        for r in used: src.locate(r["file"])
        src.stage([src.locate(r["file"]) for r in used])
        encoder=Encoder(FEATURE_MODE,FACE_MODE,XLSR_WEIGHTS,RESNET_WEIGHTS)
        cache=FeatureCache(encoder)
        ids=[r for records in selected.values() for r in records]
        fingerprint=prior_manifest_digest(selected["train"]+selected["dev"])
        media_stamp=uid([src.ident(src.locate(r["file"])) for r in selected["train"]+selected["dev"]])
        config=dict(code=CODE_VERSION,input_designation=("synthetic_self_check" if self_check else "user_supplied"),features=encoder.spec,preset=PRESET,split_seed=DATA_SEED,
                    training_manifest_hash=fingerprint,train_dev_media_stamp=media_stamp,
                    split_policy="pilot_after_quick_v1",previous_quick_manifest=PRIOR_QUICK_FINGERPRINT,
                    epochs=EPOCHS,hidden=HIDDEN,taps=TAPS,max_windows=MAX_FIT_WINDOWS,
                    shift_training=USE_SHIFT_CONSISTENCY,runtime=RUNTIME,cache_keys_version=2)
        key=uid(config)[:16]
        eval_options={"test":RUN_FINAL_TEST,"robustness":RUN_ROBUSTNESS,"seeds":SEEDS,"model":EVAL_MODEL,
                      "external_manifest":digest_file(EXTERNAL_CSV) if EXTERNAL_CSV else None,
                      "train_families":sorted(TRAIN_GENERATOR_FAMILIES),"heldout_families":sorted(HELDOUT_EXTERNAL_FAMILIES),
                      "reference_json":digest_file(BA_TFD_JSON) if BA_TFD_JSON else None,
                      "reference_method":BA_TFD_LABEL if BA_TFD_JSON else (OFFICIAL_CONFIG if OFFICIAL_REPO else None),
                      "official_checkpoint":str(OFFICIAL_CKPT) if OFFICIAL_CKPT else None,
                      "inference_file":str(NEW_VIDEO) if NEW_VIDEO else None}
        eval_options["unseen_test_manifest_hash"] = prior_manifest_digest(selected["test"]) if RUN_FINAL_TEST else None
        eval_options["holdout_verification"] = holdout
        eval_options["audit_hash"] = uid([rejected, reserved])
        folder=OUT/"runs"/(key+"_"+uid(eval_options)[:8]);folder.mkdir(parents=True,exist_ok=True)
        write_json(folder/"evaluation_plan.json",eval_options)
        write_json(folder/"holdout_audit.json",holdout)
        cols_excluded = ["metadata_row","file","split","reason","identity","duration","n_fakes","modify_audio","modify_video","fake_periods"]
        write_csv(folder/"metadata_exclusions.csv",pd.DataFrame(rejected).reindex(columns=cols_excluded))
        write_csv(folder/"prior_test_exclusions.csv",pd.DataFrame(reserved).reindex(
                columns=["file","split","source_id","speaker_id","reason"]))
        # During development publish only train/dev selection and outputs.
        ids=used
        write_json(folder/"protocol.json",config)
        write_csv(folder/"selected_manifest.csv",[dict(file=r["file"],split=r["split"],condition=r["kind"],source_id=r["group"],
                   speaker_id=r["speaker"],duration_s=r["duration"],audio_segments=json.dumps(r["a"]),visual_segments=json.dumps(r["v"])) for r in ids])
        counts=pd.DataFrame([{"split":r["split"],"condition":r["kind"],"speaker_known":bool(r["speaker"])} for r in ids])
        print("Run folder:",folder); shown(pd.crosstab(counts["split"],counts["condition"]))
        write_json(folder/"coverage.json",{"metadata_records":original_count,"metadata_rows_quarantined":len(rejected),"rows_reserved_from_prior_test":len(reserved),"eligible_records":len(base),"partial_input_filter":ALLOW_PARTIAL_INPUT,"selected":len(ids),"selected_speaker_ids_known":sum(bool(r["speaker"]) for r in ids),
                   "split_counts":counts.groupby("split").size().to_dict(),"source_map_supplied":bool(SOURCE_MAP_CSV)})
        print(f"Reading/caching {len(selected['train'])} train + {len(selected['dev'])} dev clips.", flush=True)
        warm(selected["train"]+selected["dev"],cache,src)
        all_internal=[]; all_external=[]; bundles=[]
        for seed in SEEDS:
            sd=folder/("seed_"+str(seed));sd.mkdir(exist_ok=True)
            local_store=OUT/"models"/key/sd.name
            bp=local_store/"model_bundle.zip"
            prev_store=Path(PREVIOUS_OUTPUT)/"models"/key/sd.name if PREVIOUS_OUTPUT else None
            previous=prev_store/bp.name if prev_store else None
            reload=bp if bp.is_file() else (previous if previous and previous.is_file() else None)
            if reload:
                heads,meta=open_bundle(reload)
                if meta["protocol"]!=key or meta["seed"]!=seed: raise ValueError("Mismatched saved checkpoint.")
                enforce_spec(encoder,meta)
                if reload!=bp: save_bundle(bp,heads,meta)
                devrec=record_predictions(selected["dev"],cache,src,heads)
                print("Resumed completed weights:",sd.name)
            else:
                heads,history=trained_heads(selected["train"],selected["dev"],cache,src,seed,local_store/"heads",
                                            prev_store/"heads" if prev_store else None)
                devrec=record_predictions(selected["dev"],cache,src,heads)
                thresholds=threshold_all(devrec,heads)
                meta=dict(code=CODE_VERSION,step=STEP,encoder=encoder.spec,models=MODELS,thresholds=thresholds,
                          seed=seed,protocol=key,history=history,config=config,
                          development_source_hashes=sorted({uid(["source",r["group"]]) for r in selected["train"]+selected["dev"] if r["group_verified"]}),
                          development_speaker_hashes=sorted({uid(["speaker",r["speaker"]]) for r in selected["train"]+selected["dev"] if r["speaker"]}))
                save_bundle(bp,heads,meta)
            write_csv(sd/"train_history.csv",meta["history"]); write_json(sd/"dev_thresholds.json",meta["thresholds"])
            bundles.append(bp)
            write_json(sd/"bundle_location.json",{"path_relative_to_output":str(bp.relative_to(OUT))})
            conditions=[("clean",0,False)]
            if RUN_ROBUSTNESS:
                conditions += [("matched_clean",0,False),("delay_plus240ms",3,False),("delay_minus240ms",-3,False),("compressed",0,True),
                               ("compressed_plus240ms",3,True),("compressed_minus240ms",-3,True)]
            for split in (("dev","test") if RUN_FINAL_TEST else ("dev",)):
                for name,k,coded in conditions:
                    rec=devrec if split=="dev" and name=="clean" else record_predictions(selected[split],cache,src,heads,k,coded,matched=(name!="clean"))
                    all_internal+=save_eval(sd/split/name,rec,meta["models"],meta["thresholds"],split,name,seed)
                    if name=="clean":
                        rr,ff,ss=next((entry for entry in rec if entry[0]["label"]==1),rec[0])
                        timeline(sd/split/"example_timeline.png",rr,ff,ss,EVAL_MODEL,meta)
                print("Evaluated",split,sd.name,flush=True)
            if EXTERNAL_CSV:
                devrows=selected["train"]+selected["dev"]
                all_external+=evaluate_external(EXTERNAL_CSV,encoder,cache,heads,meta,sd/"external",
                    [r["group"] for r in devrows],[r["speaker"] for r in devrows if r["speaker"]])
            if NEW_VIDEO:
                out,_=predict_clip(NEW_VIDEO,encoder,heads,meta); write_json(sd/"new_video.json",out)
        write_csv(folder/"metrics_all.csv",all_internal);summarize_seeds(folder/"seed_summary.csv",all_internal)
        if all_external:
            write_csv(folder/"external_metrics_all.csv",all_external);summarize_seeds(folder/"external_seed_summary.csv",all_external)
        reference=BA_TFD_JSON
        if OFFICIAL_REPO or OFFICIAL_CKPT:
            if not RUN_FINAL_TEST or reference or not (OFFICIAL_REPO and OFFICIAL_CKPT):
                raise ValueError("For an official run, enable final test, set both repo and checkpoint, and leave BA_TFD_JSON empty.")
            reference=run_official(src,selected["test"],folder/"official")
        if reference:
            if not RUN_FINAL_TEST: raise ValueError("Compare official predictions only after enabling the final test.")
            method=BA_TFD_LABEL if BA_TFD_JSON else ("BA-TFD+" if "plus" in str(OFFICIAL_CONFIG).lower() else "BA-TFD")
            write_csv(folder/"external_baseline_comparison.csv",[official_compare(reference,selected["test"],method)])
        cols=["seed","split","condition","model","any_AP@50","any_segment_F1@50","genuine_false_alarms","genuine_clips","genuine_FPR"]
        print("Metrics are computed from your clips. Undefined metrics appear as NaN.")
        shown(pd.DataFrame(all_internal).loc[lambda d:d.condition=="clean",cols])
        return folder,bundles
    finally:
        src.close()

def use_bundle():
    if not BUNDLE_PATH or not (NEW_VIDEO or EXTERNAL_CSV):
        raise ValueError("In predict mode, set BUNDLE_PATH and NEW_VIDEO and/or EXTERNAL_CSV.")
    heads,meta=open_bundle(BUNDLE_PATH)
    spec=meta["encoder"]
    encoder=Encoder(spec["mode"],spec["face"],XLSR_WEIGHTS,RESNET_WEIGHTS)
    enforce_spec(encoder,meta); cache=FeatureCache(encoder)
    dest=OUT/"inference"/(meta["protocol"]+"_seed"+str(meta["seed"]));dest.mkdir(parents=True,exist_ok=True)
    if NEW_VIDEO:
        info,(f,s)=predict_clip(NEW_VIDEO,encoder,heads,meta)
        write_json(dest/"new_video.json",info);timeline(dest/"new_video.png",{"file":info["file"]},f,s,info["model"],meta)
        print(json.dumps(json_ready(info),indent=2))
    if EXTERNAL_CSV:
        metric=evaluate_external(EXTERNAL_CSV,encoder,cache,heads,meta,dest/"external")
        shown(pd.DataFrame(metric).query("model == @EVAL_MODEL")[["split","clip_labeled","genuine_clips","genuine_FPR","clip_AUROC","any_AP@50"]])
    return dest,[Path(BUNDLE_PATH)]

try:
    with threadpool_limits(limits=THREADS):
        RESULTS,BUNDLES=train_project() if ACTION=="train" else use_bundle()
except Exception as err:
    print(f"STOP | {type(err).__name__}: {err}", flush=True)
    raise
print("Outputs:",RESULTS)
for file in BUNDLES: print("Model bundle:",file)
if ACTION == "train" and PRESET == "pilot":
    # One small output to download from the root of Kaggle's Output panel.
    pilot_zip = WORK / ("PILOT_RESULTS_" + RESULTS.name + ".zip")
    with atomic_path(pilot_zip) as temp:
        with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for name in ("metrics_all.csv", "metadata_exclusions.csv", "holdout_audit.json",
                         "coverage.json", "protocol.json", "evaluation_plan.json", "selected_manifest.csv"):
                archive.write(RESULTS / name, name)
            for seed, bundle in zip(SEEDS, BUNDLES):
                seed_folder = RESULTS / f"seed_{seed}"
                archive.write(bundle, f"seed_{seed}/model_bundle.zip")
                archive.write(seed_folder / "dev" / "example_timeline.png", f"seed_{seed}/dev_timeline.png")
    print("Pilot ZIP:", pilot_zip, flush=True)
print("DONE | " + ACTION + " | outputs saved", flush=True)

## 11 · Own output 

In [ ]:
print("[11/11] Displaying saved results ...", flush=True)
if ACTION == "train":
    seed_folder = RESULTS / ("seed_" + str(SEEDS[0]))
    figure = seed_folder / "dev" / "example_timeline.png"
    coverage = json.loads((RESULTS / "coverage.json").read_text())
    print("Selected split counts:", coverage["split_counts"])
    print("Review:", RESULTS / "metrics_all.csv")
    print("Thresholds:", seed_folder / "dev_thresholds.json")
    print("Saved model:", BUNDLES[0])
else:
    figure = RESULTS / "new_video.png" if NEW_VIDEO else None
if figure and figure.is_file():
    try:
        from IPython.display import Image as NotebookImage, display as nb_display
    except ImportError:
        print("Timeline figure:", figure)
    else:
        nb_display(NotebookImage(filename=str(figure)))